In [ ]:
import plotly.io as pio
import json

# 加载Plotly JSON
with open("my_plot.json", "r", encoding="utf-8") as f:
    fig_dict = json.load(f)

# 生成图像（如 PNG 缩略图）
img_bytes = pio.to_image(fig_dict, format="png", width=800, height=600)

# 保存为文件
with open("thumbnail.png", "wb") as f:
    f.write(img_bytes)

In [ ]:
def quick_sanitize(fig_dict: dict) -> dict:
    """
    极简宽容清洗：把常见会触发校验的问题值改成更安全的 None/删除。
    不改变可视效果（浏览器本来也会fallback）。
    """
    def _fix(o):
        if isinstance(o, dict):
            # 1) 空字符串字体 → None（浏览器会用默认字体）
            if "font" in o and isinstance(o["font"], dict):
                if o["font"].get("family", None) == "":
                    o["font"]["family"] = None
            # 2) 空字符串色值 → 删除（让 Plotly 用默认色）
            for k in list(o.keys()):
                if (k.endswith("color") or k.endswith("colors")) and o[k] == "":
                    o.pop(k)
            # 3) None/空串的数值字段 → 删除（如 linewidth、size 等）
            for k in list(o.keys()):
                if o[k] in ("", None) and k in {"size","width","height","linewidth","line_width"}:
                    o.pop(k)
            # 递归
            for v in o.values():
                _fix(v)
        elif isinstance(o, list):
            for x in o:
                _fix(x)

    fig_copy = json.loads(json.dumps(fig_dict))  # 深拷贝
    _fix(fig_copy)
    return fig_copy

In [ ]:
import plotly.io as pio
import json
import plotly.graph_objs as go

# 加载Plotly JSON
with open("KEGG_002306_813.json", "r", encoding="utf-8") as f:
    fig_dict = json.load(f)

fig = go.Figure(fig_dict, skip_invalid=True)

# 生成图像（如 PNG 缩略图）
img_bytes = pio.to_image(fig, format="webp", width=800, height=600, validate=False)

# 保存为文件
with open("thumbnail.800.2.webp", "wb") as f:
    f.write(img_bytes)

In [ ]:
import plotly.io as pio
import json
import plotly.graph_objs as go



In [ ]:
import plotly.io as pio
import json
import copy

def scale_plotly_figure(fig_dict, scale=0.5):
    layout = fig_dict.get("layout", {})
    layout = copy.deepcopy(layout)

    # 缩放字体
    if "font" in layout:
        if "size" in layout["font"]:
            layout["font"]["size"] = layout["font"]["size"] * scale

    # 缩放 legend 字体
    if "legend" in layout and "font" in layout["legend"]:
        if "size" in layout["legend"]["font"]:
            layout["legend"]["font"]["size"] *= scale

    # 缩放 axis 字体
    for axis in ['xaxis', 'yaxis']:
        if axis in layout and "title" in layout[axis]:
            if isinstance(layout[axis]["title"], dict) and "font" in layout[axis]["title"]:
                if "size" in layout[axis]["title"]["font"]:
                    layout[axis]["title"]["font"]["size"] *= scale
        if axis in layout and "tickfont" in layout[axis]:
            if "size" in layout[axis]["tickfont"]:
                layout[axis]["tickfont"]["size"] *= scale

    # 缩放图尺寸
    if "width" in layout:
        layout["width"] = int(layout["width"] * scale)
    if "height" in layout:
        layout["height"] = int(layout["height"] * scale)

    fig_dict["layout"] = layout
    return fig_dict

In [ ]:
import io
from PIL import Image

buffer = io.BytesIO(img_bytes)
    # 用 PIL 打开该图像
img = Image.open(buffer)
    # 可选：转换为 RGBA 模式，保持透明
    # img = img.convert("RGBA")
    # 复制图像用于缩略图处理
thumb = img.copy()
thumb.thumbnail((200, 150), resample=Image.LANCZOS)
thumb.save("thumbnail.small.png")

In [ ]:
buffer = io.BytesIO(img_bytes)  # img_bytes 来自 plotly.to_image
img = Image.open(buffer)
thumb = img.copy()
thumb.thumbnail((800, 600), resample=Image.LANCZOS)
thumb.save("thumbnail.800.q95.webp", quality=95)  # 无损 WebP 保存

In [ ]:
buffer = io.BytesIO(img_bytes)  # img_bytes 来自 plotly.to_image
img = Image.open(buffer)
thumb = img.copy()
thumb.thumbnail((400, 300), resample=Image.LANCZOS)
thumb.save("thumbnail.400.q95.webp", quality=95) 

In [ ]:
from PIL import Image

def generate_png_thumbnail(input_path, output_path, max_size=(400, 300), resample=Image.LANCZOS):
    with Image.open(input_path) as img:
        # 复制图像对象，防止修改源图
        thumb = img.copy()
        # 等比缩放至 max_size 不拉伸，最大不超过指定宽高
        thumb.thumbnail(max_size, resample=resample)
        thumb.save(output_path, quality=70)
    print(f"✅ 缩略图已生成: {output_path}, 尺寸: {thumb.size}")

In [ ]:
generate_png_thumbnail("thumbnail.png", "thumbnail.400.png", (400, 300))

In [ ]:
generate_png_thumbnail("screenshot-20250917-183432.png", "network.thumbnail.800.webp", (800, 600))

In [ ]:
!ls -hs

In [ ]:
import plotly.io as pio
import json
import copy
import os

def scale_plotly_figure_to_size(fig_dict, target_width=400, target_height=300):
    layout = fig_dict.get("layout", {})
    layout = copy.deepcopy(layout)

    original_width = layout.get("width", 700)   # 默认 Plotly 宽高
    original_height = layout.get("height", 450)

    scale_x = target_width / original_width
    scale_y = target_height / original_height
    scale = min(scale_x, scale_y)  # 保证不拉伸，等比缩放

    # 缩放字体
    if "font" in layout and "size" in layout["font"]:
        layout["font"]["size"] *= scale

    # 缩放 legend 字体
    if "legend" in layout and "font" in layout["legend"]:
        if "size" in layout["legend"]["font"]:
            layout["legend"]["font"]["size"] *= scale

    # 缩放坐标轴字体
    for axis in ['xaxis', 'yaxis']:
        if axis in layout:
            if "title" in layout[axis] and "font" in layout[axis]["title"]:
                if "size" in layout[axis]["title"]["font"]:
                    layout[axis]["title"]["font"]["size"] *= scale
            if "tickfont" in layout[axis] and "size" in layout[axis]["tickfont"]:
                layout[axis]["tickfont"]["size"] *= scale

    # 应用目标尺寸
    layout["width"] = target_width
    layout["height"] = target_height

    fig_dict["layout"] = layout
    return fig_dict

def generate_thumbnail(json_path, output_path, target_width=400, target_height=300, fmt="png"):
    with open(json_path, "r", encoding="utf-8") as f:
        fig_dict = json.load(f)

    # 缩放图表以适配目标尺寸
    scaled_fig = scale_plotly_figure_to_size(fig_dict, target_width, target_height)

    # 渲染为图像字节
    img_bytes = pio.to_image(scaled_fig, format=fmt, width=target_width, height=target_height)

    # 保存
    with open(output_path, "wb") as f:
        f.write(img_bytes)

    print(f"✅ 缩略图已生成: {output_path}  尺寸: {target_width}x{target_height}")

In [ ]:
def generate_thumbnail(json_path, output_path, target_width=400, target_height=300, fmt="png"):
    import plotly.io as pio
    import json, copy

    with open(json_path, "r", encoding="utf-8") as f:
        fig_dict = json.load(f)

    layout = copy.deepcopy(fig_dict.get("layout", {}))
    original_width = layout.get("width", 700)
    original_height = layout.get("height", 450)

    # 缩放比例
    scale = min(target_width / original_width, target_height / original_height)

    def scale_font(font_dict):
        if isinstance(font_dict, dict) and "size" in font_dict:
            font_dict["size"] = max(1, font_dict["size"] * scale)
        return font_dict

    # 缩放字体、legend、xaxis/yaxis
    if "font" in layout:
        layout["font"] = scale_font(layout["font"])

    if "legend" in layout and "font" in layout["legend"]:
        layout["legend"]["font"] = scale_font(layout["legend"]["font"])

    for axis in ['xaxis', 'yaxis']:
        ax = layout.get(axis, {})
        if "title" in ax and "font" in ax["title"]:
            ax["title"]["font"] = scale_font(ax["title"]["font"])
        if "tickfont" in ax:
            ax["tickfont"] = scale_font(ax["tickfont"])
        layout[axis] = ax

    # 缩放图尺寸
    layout["width"] = target_width
    layout["height"] = target_height
    fig_dict["layout"] = layout

    # 生成缩略图
    img_bytes = pio.to_image(fig_dict, format=fmt, width=target_width, height=target_height)
    with open(output_path, "wb") as f:
        f.write(img_bytes)
    print(f"✅ 缩略图已生成：{output_path}，按比例缩放至 {target_width}×{target_height}")

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
import json
import copy

def generate_thumbnail(json_path, output_path, target_width=400, target_height=300, fmt="png"):
    # 读取 JSON 文件
    with open(json_path, "r", encoding="utf-8") as f:
        fig_dict = json.load(f)

    # 获取原始尺寸（如未设置则使用默认）
    layout = fig_dict.get("layout", {})
    original_width = layout.get("width", 700)
    original_height = layout.get("height", 450)

    # 计算缩放比例（保持等比）
    scale = min(target_width / original_width, target_height / original_height)

    # 深拷贝 layout 防止修改原数据
    layout = copy.deepcopy(layout)

    # 缩放字体、legend、坐标轴文字
    def scale_font(font_dict):
        if isinstance(font_dict, dict) and "size" in font_dict:
            font_dict["size"] = max(1, font_dict["size"] * scale)
        return font_dict

    if "font" in layout:
        layout["font"] = scale_font(layout["font"])

    if "legend" in layout and "font" in layout["legend"]:
        layout["legend"]["font"] = scale_font(layout["legend"]["font"])

    for axis in ['xaxis', 'yaxis']:
        ax = layout.get(axis, {})
        if "title" in ax and "font" in ax["title"]:
            ax["title"]["font"] = scale_font(ax["title"]["font"])
        if "tickfont" in ax:
            ax["tickfont"] = scale_font(ax["tickfont"])
        layout[axis] = ax

    # 设置新的尺寸
    layout["width"] = target_width
    layout["height"] = target_height
    fig_dict["layout"] = layout

    # 使用 go.Figure 保证 layout 被正确应用
    fig = go.Figure(fig_dict)

    # 生成缩略图
    img_bytes = pio.to_image(fig, format=fmt)
    with open(output_path, "wb") as f:
        f.write(img_bytes)

    print(f"✅ 缩略图已生成: {output_path} （{target_width}×{target_height}，缩放比例：{scale:.2f}）")

In [ ]:
generate_thumbnail("my_plot.json", "thumbnail2.png", target_width=400, target_height=300, fmt="png")

In [ ]:
!ls -sh